<a href="https://colab.research.google.com/github/NadiiaStuzhuk/DI-Bootcamp-Stage1/blob/main/Daily_Challenge_How_to_Finetune_LLMs_with_LoRA_Student_NS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step 1: Install Required Libraries & Create Cache Directory

%pip install -q peft==0.4.0 datasets accelerate transformers
!mkdir -p ../cache/working

# Step 2 & 3: Load Pre-trained Model, Tokenizer, and Preprocess Dataset

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

# 2. Load the pre-trained language model (bigscience/bloomz-560m) and its tokenizer
model_name = "bigscience/bloomz-560m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

# 3. Load a 10% sample of the training split of "Abirate/english_quotes"
data = load_dataset("Abirate/english_quotes", split="train[:10%]")

# Tokenize the "quote" column
data = data.map(lambda samples: tokenizer(samples["quote"]), batched=True)

# Select a small subset for quick inspection / demonstration
train_sample = data.select(range(5))
display(train_sample)

# Step 4 & 5: Configure LoRA and Apply to Foundation Model
# For bigscience/bloomz-560m (BLOOM architecture), the attention projection layers are combined into a single module named "query_key_value".

import peft
from peft import LoraConfig, get_peft_model

# 4. Configure LoRA using LoraConfig
lora_config = LoraConfig(
    r=1,                                  # Rank of the low-rank decomposition matrices
    lora_alpha=1,                         # Scaling factor adjusting the weight matrix magnitude
    target_modules=["query_key_value"],   # BLOOM attention layer module name
    lora_dropout=0.05,                    # Dropout probability for LoRA layers
    bias="none",                          # Do not train bias parameters
    task_type="CAUSAL_LM"                 # Causal Language Modeling task
)

# 5. Apply LoRA adapter layers to the foundation model
peft_model = get_peft_model(foundation_model, lora_config)
peft_model.print_trainable_parameters()

# Step 6 & 7: Set Up TrainingArguments and Train with Trainer
import os
import transformers
from transformers import TrainingArguments, Trainer

output_directory = os.path.join("../cache/working", "peft_lab_outputs")

# 6. Set up training arguments
training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    auto_find_batch_size=True,
    learning_rate=3e-2,      # Higher learning rate suitable for LoRA adapters
    num_train_epochs=3,      # Number of training epochs
    use_cpu=True             # Set to False if running on a Colab GPU runtime
)

# 7. Initialize and train the model using Trainer
# Note: Use `train_dataset=data` for the full 10% split, or `train_dataset=train_sample` for rapid CPU testing
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=data,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

trainer.train()

# Step 8, 9 & 10: Save, Reload, and Perform Inference with the Fine-Tuned LoRA Model
import time
import torch
from peft import PeftModel

# 8. Save the fine-tuned LoRA model with a timestamp
time_now = int(time.time())
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")
trainer.model.save_pretrained(peft_model_path)
print(f"LoRA adapter weights saved to: {peft_model_path}")

# 9. Load the saved LoRA model for inference (is_trainable=False prevents further gradient updates)
loaded_peft_model = PeftModel.from_pretrained(
    foundation_model,
    peft_model_path,
    is_trainable=False
)
loaded_peft_model.eval()

# 10. Generate text using the fine-tuned model and tokenizer
device = next(loaded_peft_model.parameters()).device
inputs = tokenizer("Two things are infinite: ", return_tensors="pt").to(device)

with torch.no_grad():
    outputs = loaded_peft_model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=32,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7
    )

print("\n--- Generated Quote ---")
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

# Key Technical Notes
# target_modules=["query_key_value"]: Unlike LLaMA or GPT-NeoX (which separate q_proj, k_proj, and v_proj), the BLOOM architecture fuses query, key, and value projections into a single linear layer called query_key_value. Targeting this module with r=1 trains only 98,304 parameters out of 559,312,896 total parameters (~0.0176% of the model).

# CPU vs. GPU Execution: The template specifies use_cpu=True. If you run the full 10% dataset (~250 quotes) on CPU, training may take several minutes; passing train_dataset=train_sample (the 5-row subset) completes in a few seconds, or you can switch use_cpu=False if a T4 GPU is enabled in Google Colab.

config.json:   0%|          | 0.00/715 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 14.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]